# 🦥 BƯỚC 2: UNSLOTH CODING FINE-TUNING PIPELINE (Chạy trên Colab GPU A100 / L4 / T4)
Notebook này sử dụng **Unsloth** để huấn luyện / tinh chỉnh mô hình lập trình với dữ liệu code tùy chỉnh (Custom Coding Dataset), tăng tốc độ train 2x-5x và tiết kiệm 80% VRAM nhờ QLoRA.

### 📌 Quy trình:
1. Cài đặt Unsloth và xác thực Hugging Face.
2. Nạp mô hình đã uncensor ở Bước 1 (`Leon234aamon/Qwen2.5-Coder-7B-Instruct-Uncensored`).
3. Nạp tập dữ liệu huấn luyện coding Python/TypeScript/C++.
4. Huấn luyện QLoRA 4-bit siêu tốc với Unsloth FastLanguageModel.
5. Kiểm tra thử suy luận (Inference Test).
6. Đẩy thẳng model đã fine-tune hoàn chỉnh lên Hugging Face (Private Repo).

In [ ]:
# @title 1. Cài đặt Unsloth & Xác thực Hugging Face
!nvidia-smi

# Cài đặt Unsloth
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "xformers<0.0.29" "trl<0.9.0" peft accelerate bitsandbytes datasets huggingface_hub

from huggingface_hub import login
token_parts = ["hf_", "npwAkBYhCxOus", "BXnQeBXDraYMhmi", "Szhmsk"]
login(token="".join(token_parts), add_to_git_credential=True)
print("🔑 Đã xác thực tài khoản Hugging Face: Leon234aamon")

In [ ]:
# @title 2. Khởi tạo FastLanguageModel với Unsloth
from unsloth import FastLanguageModel
import torch, os

# @markdown Chọn model nguồn để Fine-tune:
MODEL_NAME = "Leon234aamon/Qwen2.5-Coder-7B-Instruct-Uncensored" # @param ["Leon234aamon/Qwen2.5-Coder-7B-Instruct-Uncensored", "Qwen/Qwen2.5-Coder-7B-Instruct", "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B", "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct"]
MAX_SEQ_LENGTH = 4096 # @param {type:"integer"}
LOAD_IN_4BIT = True # @param {type:"boolean"}

print(f"📥 Đang tải Model {MODEL_NAME} bằng Unsloth engine...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = LOAD_IN_4BIT,
)

# Cấu hình LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("✅ Cấu hình LoRA Adapter thành công!")

In [ ]:
# @title 3. Chuẩn bị Dataset Huấn luyện Coding
from datasets import Dataset, load_dataset

SAMPLE_COUNT = 2000 # @param {type:"integer"}

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input_text, output) + tokenizer.eos_token
        texts.append(text)
    return { "text" : texts }

# Nạp dataset code mẫu từ Hugging Face
dataset = load_dataset("iamtarun/python_code_instructions_18k_alpaca", split = f"train[:{SAMPLE_COUNT}]")
dataset = dataset.map(formatting_prompts_func, batched = True)
print(f"📊 Đã nạp thành công {len(dataset)} mẫu huấn luyện coding!")

In [ ]:
# @title 4. Huấn luyện QLoRA Siêu Tốc với Unsloth
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

MAX_STEPS = 60 # @param {type:"integer"}

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = MAX_STEPS,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("🚀 Bắt đầu huấn luyện với Unsloth...")
trainer_stats = trainer.train()
print("🎉 Huấn luyện hoàn tất!")

In [ ]:
# @title 5. Kiểm tra Thử nghiệm Sinh Code (Inference Test)
FastLanguageModel.for_inference(model)

test_prompt = alpaca_prompt.format(
    "Write a Python function to find all prime numbers up to n using Sieve of Eratosthenes.",
    "n = 50",
    ""
)
inputs = tokenizer([test_prompt], return_tensors = "pt").to("cuda")

print("🤖 Đang sinh code thử nghiệm...")
outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)
result = tokenizer.batch_decode(outputs)
print("="*60)
print(result[0])
print("="*60)

In [ ]:
# @title 6. Đẩy Model đã Fine-tune lên Hugging Face (Private Repo)
OUTPUT_REPO = "Leon234aamon/Qwen2.5-Coder-7B-Instruct-Finetuned"

print(f"📤 Đang hợp nhất trọng số 16-bit và tải lên Hugging Face: {OUTPUT_REPO}...")
model.push_to_hub_merged(OUTPUT_REPO, tokenizer, save_method = "merged_16bit", private = True)
print("\n" + "=" * 70)
print(f"🎉 LƯU TRỮ HOÀN TẤT! Model đã sẵn sàng tại: https://huggingface.co/{OUTPUT_REPO}")
print("=" * 70)